# Generators


## Lazy Pipeline over a Large File

Generators chain transformations without ever storing the intermediate data. The next cell creates a log of 500,000 requests (`number;user;duration_ms`) and provides `peak_memory`, which measures the peak memory of a call.

1. Write three generators: `read_lines(path)`, which yields the lines of the file, `parse(lines)`, which yields `(user, duration)` tuples, and `slow(records, threshold)`, which only keeps requests longer than `threshold`.
2. Write `total_slow_generators(path)`, which chains your generators and returns, with `sum`, the total duration of the requests longer than 900 ms.
3. Write `total_slow_lists(path)`, which computes the same thing but stores each step in a list (`readlines`, `list(...)`).
4. Compare the results and the peak memory of both versions with `peak_memory`. The totals must be equal, and the list-based version should use tens of megabytes more.

In [ ]:
import random
import tracemalloc

with open("requests.log", "w", encoding="utf8") as fh:
    for i in range(500_000):
        fh.write(f"{i};user{random.randrange(1_000)};{random.randrange(1_000)}\n")


def peak_memory(function):
    """Run function() and return its result and its peak memory in MB."""
    tracemalloc.start()
    result = function()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, peak / 1e6

In [ ]:
# Your code here

### Solution

In [ ]:
from collections.abc import Iterable, Iterator


def read_lines(path: str) -> Iterator[str]:
    with open(path, encoding="utf8") as fh:
        yield from fh


def parse(lines: Iterable[str]) -> Iterator[tuple[str, int]]:
    for line in lines:
        _, user, duration = line.rstrip("\n").split(";")
        yield user, int(duration)


def slow(
    records: Iterable[tuple[str, int]], threshold: int
) -> Iterator[tuple[str, int]]:
    for user, duration in records:
        if duration > threshold:
            yield user, duration


def total_slow_generators(path: str) -> int:
    return sum(duration for _, duration in slow(parse(read_lines(path)), 900))


def total_slow_lists(path: str) -> int:
    with open(path, encoding="utf8") as fh:
        lines = fh.readlines()
    records = list(parse(lines))
    return sum(duration for _, duration in list(slow(records, 900)))


total_g, peak_g = peak_memory(lambda: total_slow_generators("requests.log"))
total_l, peak_l = peak_memory(lambda: total_slow_lists("requests.log"))
print(total_g == total_l, f"generators: {peak_g:.1f} MB, lists: {peak_l:.1f} MB")

The generators only keep one line in memory at a time, whatever the size of the file: the list-based version stores the whole file, then every tuple.

## Infinite Date Generator

Write the function `gen_dates` that, starting from a given date, generates all successive dates separated by one week.


In [ ]:
import datetime
from collections.abc import Iterator


def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  pass  # Your code here

Propose a direct way to use this infinite generator.

Also propose a method `take[T](iterable: Iterable[T], n: int) -> Iterator[T]` (which takes an iterable as input and returns its first `n` elements) to use this infinite generator.

In [ ]:
# Your code here

### Solution

In [ ]:
import datetime
from collections.abc import Iterable, Iterator


def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  week = datetime.timedelta(days=7)
  while True:
    date += week
    yield date


# With the walrus operator
def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  week = datetime.timedelta(days=7)
  while True:
    yield (date := date + week)


for i, d in enumerate(gen_dates(datetime.date(1900, 1, 1))):
  if i == 10:
    break
  print(d)


def take[T](iterable: Iterable[T], n: int) -> Iterator[T]:
  for i, item in enumerate(iterable):
    if i == n:
      break
    yield item


def take[T](iterable: Iterable[T], n: int) -> Iterator[T]:
  for item, _ in zip(iterable, range(n)):
    yield item


for date in take(gen_dates(datetime.date(1900, 1, 1)), 10):
  print(date)

## Releasing a Resource with `close` (hard)

A generator can hold a resource (file, connection…) while it yields its values.

Write a generator `numbered_lines(path)` that prints `opening`, opens the file and yields its numbered lines `(number, line)` with `yield from`, then prints `closing` when the file is closed again, including when the caller stops before the end.

Read the first three lines of `requests.log` with `next`, then call `close()` on the generator: `closing` must be printed right away. Why?

In [ ]:
# Your code here

### Solution

In [ ]:
from collections.abc import Iterator


def numbered_lines(path: str) -> Iterator[tuple[int, str]]:
    print("opening")
    with open(path, encoding="utf8") as fh:
        try:
            yield from enumerate(fh, start=1)
        finally:
            print("closing")


lines = numbered_lines("requests.log")
for _ in range(3):
    print(next(lines))
lines.close()

`close()` raises `GeneratorExit` at the generator's pause point: the `finally` block runs and the `with` closes the file. Without `close()`, the file would only be closed when the generator is destroyed.

## Using the `send` method of a generator (very advanced)

Design a generator that, given an initial sum of money, will receive an interest rate from you and return the new value of the sum after applying the rate.


In [ ]:
# Your code here

### Solution

In [ ]:
from collections.abc import Generator


def interests(amount: float) -> Generator[float, float, None]:
  while True:
    rate = yield amount
    amount *= rate
    # Or directly
    # amount *= yield amount


amounts = interests(100)
print(next(amounts))  # Generator initialization
for rate in [1.05, 1.08, 0.98]:
  print(amounts.send(rate))